## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-15-23-45-01 +0000,chitri,Officials say a tanker’s steering failed befor...,https://www.chicagotribune.com/2026/11/15/tank...
1,2026-11-15-23-43-54 +0000,startribune,P.J. Fleck fires Gophers offensive coordinator...,https://www.startribune.com/minnesota-gophers-...
2,2026-11-15-23-40-27 +0000,nypost,Gang of kids batter boy at Beverly Hills High ...,https://nypost.com/2026/11/15/us-news/gang-of-...
3,2026-11-15-23-35-02 +0000,cbc,Top scorer of South Korea tax accounting exam ...,https://www.cbc.ca/news/world/south-korea-tax-...
4,2026-11-15-23-33-20 +0000,nypost,NYS Dems win big after changing laws for local...,https://nypost.com/2026/11/15/us-news/nys-dems...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2445/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
103,trump,20
94,ground,14
7,new,13
101,nvidia,11
460,huang,10


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
93,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...,60
123,2026-11-15-15-17-32 +0000,chitri,Transportation head vows to find ‘root cause’ ...,https://www.chicagotribune.com/2026/11/15/trum...,60
121,2026-11-15-15-37-44 +0000,wapo,Trump blames Biden for FAA problems in first c...,https://www.washingtonpost.com/politics/2026/1...,59
185,2026-11-15-09-23-51 +0000,nypost,‘SNL’ roasts ex-Nvidia boss Jensen Huang over ...,https://nypost.com/2026/11/15/business/snl-roa...,56
109,2026-11-15-17-10-54 +0000,nypost,Trump blames Biden admin for FAA outage debacl...,https://nypost.com/2026/11/15/us-news/trump-bl...,56


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
93,60,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...
123,60,2026-11-15-15-17-32 +0000,chitri,Transportation head vows to find ‘root cause’ ...,https://www.chicagotribune.com/2026/11/15/trum...
78,40,2026-11-15-19-30-51 +0000,chitri,Israel kills a top Hamas commander as Jordan s...,https://www.chicagotribune.com/2026/11/15/isra...
109,36,2026-11-15-17-10-54 +0000,nypost,Trump blames Biden admin for FAA outage debacl...,https://nypost.com/2026/11/15/us-news/trump-bl...
207,32,2026-11-15-02-53-00 +0000,wsj,A federal judge temporarily barred the Trump a...,https://www.wsj.com/politics/policy/trump-admi...
168,31,2026-11-15-10-30-00 +0000,wsj,High mortgage rates have crushed homeowners’ b...,https://www.wsj.com/economy/housing/theyre-luc...
164,30,2026-11-15-10-34-22 +0000,nypost,Vagrant sets up City Hall encampment along Mam...,https://nypost.com/2026/11/15/us-news/mamdanis...
29,28,2026-11-15-22-04-16 +0000,nypost,Long Island Dem boss under FBI probe tries to ...,https://nypost.com/2026/11/15/us-news/long-isl...
193,28,2026-11-15-05-51-30 +0000,nypost,Bahrain issues first threat alerts since Septe...,https://nypost.com/2026/11/15/world-news/iran-...
60,25,2026-11-15-20-30-00 +0000,wapo,"Southeast Asian leaders kick-start summit, wit...",https://www.washingtonpost.com/world/2026/11/1...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
